# Load Dataset

In [1]:
import pandas as pd

df = pd.read_csv("./data/download/영양.csv")

print(f"(전체 데이터 수, 전체 컬럼 수): {df.shape}")
df.head(2)

(전체 데이터 수, 전체 컬럼 수): (66, 5)


,video_url,video_id,title,channel,caption
0,https://www.youtube.com/watch?v=-LqEr-PlCz0&li...,-LqEr-PlCz0,#25저지방우유먹기,하정훈의 삐뽀삐뽀 119 소아과 (삐뽀삐뽀 119 소아과),[음악] tel 안녕하세요 소아청소년과 전문의 가져온 입니다 저지방우유 모이라고 하...
1,https://www.youtube.com/watch?v=-LqEr-PlCz0,-LqEr-PlCz0,#25저지방우유먹기,하정훈의 삐뽀삐뽀 119 소아과 (삐뽀삐뽀 119 소아과),[음악] tel 안녕하세요 소아청소년과 전문의 가져온 입니다 저지방우유 모이라고 하...


# Drop duplicates

In [2]:
df.drop_duplicates(inplace=True, subset=['video_id'])

print(f"(전체 데이터 수, 전체 컬럼 수): {df.shape}")

(전체 데이터 수, 전체 컬럼 수): (65, 5)


# Cleaning Caption

In [3]:
def cleaning_caption(data:str) -> str:
  return data.replace(" 으 ", "").replace(" 아 ", "")\
    .replace("[음악]", "").replace("[박수]", "").strip()


In [4]:
df['caption'] = df['caption'].map(cleaning_caption)


In [5]:
df.loc[0,['caption']].values

array(['tel 안녕하세요 소아청소년과 전문의 가져온 입니다 저지방우유 모이라고 하면 우리 아이가 비만 다 인데 왜 저지방우유 보도에 해요 라고 반문 하시는 부모님들이 많습니다 심지어는 어린이 업무를 먹지 않으면 큰 야크 참 생각하시는 부모도 있으십니다 우유에 대한 기본적인 지식을 알려 드리면 위는 건강의 좋은 음식입니다 편한 때부터 엄마 젖 먹고 자라는데 적어도 2 까지는 모유를 먹이는 것이 좋습니다 그런데 돌이 지나면서 모유나 분유를 중단할 아이들은 보통 우의를 묻게 됩니다 흔히 세무 이라 말한 말은 보통 이 붙게 되는데 우리는 우리 몸에 좋은 건물 담백하고 칼슘이 풍부한 음식입니다 아이들은 보통 하루에 2컵 정도의 우리를 먹는 것이 건강에 좋습니다 그런데 위의 는 지방도 많이 들어 있는데 위의 들어있는 지방은 삼겹살 소개드리는 지방 만큼이나 우리 몸에 나쁜 포화지방 입니다 포화지방을 많이 먹으면 나중에 비만 고혈압 심장병 같은 성인병이 잘 끌릴 수 가 있습니다 그래서 위에서 지방을 주의 누이가 개발되었는데 이것을 저 지방 누이 라고 합니다 지방을 다 뺀 것도 있는데 이것은 무지방 물이라고 합니다 그럼 아이들에게는 어떤 우유를 먹여야 하나 이게 공유 하실 겁니다 2 2 잊은 의 아이들은 보통 의를 먹는 것이 기본적으로 권장됩니다 두돌이 걔네 아이들에게는 지방이 두뇌 발달이 필수 길로 필요하기 때문에 저지방 우유를 먹이면 골라야 합니다 나이가 들면서 우리 몸이 필요로 하는 지방의 양이 서서히 줄기 때문에 두 돌 쯤 되는 보통의 아이들은 저지방 이나 무지방 의를 무리는 것이 권장됩니다 하루에 한 두껍 정도 의를 문의는 것이 그 응당한 인생에 큰 도움이 됩니다 2 2 부터는 저 방이라 무지방 우유를 먹이는 것이 중요한데 늦어도 5 얼 등 까지는 무지방 위로받고 무기 시작하는 것이 좋습니다 간혹 저지방 우유는 비만이 아이들 맞먹는다고 잘못 생각하시는 사람도 있는데 저지방 의 를 모는 가장 중요한 요인은 성인병 예방 입니다 물론 비만인 사람은 몸무게 때문에 도

# LLM을 이용한 Caption 클린징 

## LLM

In [6]:
from dotenv import load_dotenv 

load_dotenv()

True

In [7]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5-mini",
    reasoning_effort="high",        # 논리성 강화
)

## ChatPromptTemplate

In [8]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
"""
너는 영상 캡션 데이터를 정제하는 전문가다.

아래 문장은 음성 인식 또는 자동 생성된 캡션이라
깨진 글자, 반복 표현, 의미 없는 소리, 불필요한 기호가 포함되어 있다.

작업 목표:
- 의미는 유지한다
- 자연스럽고 올바른 한국어 문장으로 복원한다
- 새 정보를 추가하지 않는다
- 설명하지 말고 결과 문장만 출력한다

반드시 지켜야 할 치환 규칙:
- "삐뽀삐뽀119소아과" → "소아청소년과"
- "하정훈" → "소아청소년과 전문의"

주의:
- 위 치환은 문맥과 상관없이 **항상 적용**
- 다른 고유명사는 임의로 바꾸지 않는다
- 치환 사실을 설명하거나 주석을 달지 않는다

입력 문장:
{caption}

정제된 문장:
"""
)


## Chain

In [9]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

## Cleaning caption

In [10]:
captions = df["caption"].tolist()

print(f"captions: {len(captions)}")

captions: 65


> Batch 처리용 입력 준비

In [11]:
from itertools import compress


# 유효한 문장만 LLM에 전달
valid_mask = [
    isinstance(c, str) and c.strip()
    for c in captions
]

inputs = [
    {"caption": caption}
    for caption in compress(captions, valid_mask)
]

> Batch + 병렬 실행

In [12]:
from tqdm import tqdm

BATCH_SIZE = 50
MAX_CONCURRENCY = 8 # 최대 8개의 요청을 동시에 날림

def chunks(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

results = []

for chunk in tqdm(list(chunks(inputs, BATCH_SIZE))):
    results.extend(
        chain.batch(
            chunk,
            config={"max_concurrency": MAX_CONCURRENCY}
        )
    )

100%|██████████| 2/2 [22:49<00:00, 684.82s/it]


> 결과를 원래 DataFrame에 매핑

In [13]:
cleaned_texts = []
result_idx = 0

for is_valid in valid_mask:
    if is_valid:
        cleaned_texts.append(results[result_idx])
        result_idx += 1
    else:
        cleaned_texts.append(None)

df["clean_caption"] = cleaned_texts

In [14]:
df["clean_caption"] = df["clean_caption"].map(lambda x: x.strip())

In [15]:
df[['clean_caption', 'caption']].head()

,clean_caption,caption
0,"안녕하세요, 소아청소년과 전문의입니다. 저지방 우유를 권하면 '우리 아이가 비만도 ...",tel 안녕하세요 소아청소년과 전문의 가져온 입니다 저지방우유 모이라고 하면 우리 ...
2,안녕하세요. 소아청소년과 전문의입니다. 과일과 채소가 건강에 좋다는 것은 누구나 아...,222 안녕하세요 소아청소년과 전문의 가정입니다 과일과 채소가 금강이 좋다는 것은 ...
3,"안녕하세요, 소아청소년과 전문의입니다. 이유식 시작 시기에 대해 문의하시는 분들이 ...",오 오 오 예 안녕하세요 어떠한 전술과 전문의 가져옵니다 이유식 시기에 대해서 물이...
4,"안녕하세요, 소아청소년과 전문의입니다. 요즘 부모님들이 아이에게 좋은 것이 없냐고 ...",아 5 아내가 되어 소아청소년과 전문의가 정문입니다 요즘은 건강이 책을 가 돌아서 ...
5,"안녕하세요, 소아정신과 전문의입니다. 돌 무렵에 우유를 바꿔야 하는지 질문하시는 분...",on lee 안녕하세요 소아정신과 전문의 가져옵니다 돌이라는 어떤 울 묻어야 되냐 ...


In [16]:
df[['clean_caption', 'caption']].tail()

,clean_caption,caption
61,안녕하세요. 보리차는 언제부터 먹일 수 있나요? 아기 분유에 섞어도 되나요? 열이 ...,안녕하세요 서창 성과 전문의 가정입니다 보리차는 언제부터 모일 수 있나요 아기용 부...
62,안녕하세요. 이하정입니다. 요즘 부모님들은 채소와 과일을 익혀서 주는 경우가 많습니...,안녕하세요 4층 선까지 문 이하정 됩니다 유씨가 때 채소와 과일을 익히는 경우가 많...
63,"안녕하세요, 소아청소년과 전문의입니다. 요즘 이유식을 잘 먹지 않는 아이들이 많습니...",으으안녕하세요 소아청소년과 전문의 가져옵니다 휴식을 잘 안먹는 달아 아이들이 재미 ...
64,안녕하세요. 소아청소년과 전문의입니다. 오늘은 식습관에 대해 말씀드리겠습니다. 잘 ...,안녕하세요 4 청순과 전문의 하정훈 입니다 노 우드 특이한 이야기 1 아트 말씀을 ...
65,로운맘입니다. 오늘은 생후 9개월부터 14개월까지 두 발로 걷는 시기의 아기를 위한...,백야 세럼 맘입니다 오늘은 생후 9개월 부터 14개월 까지 두 발로 걷는 시기에 아...


In [17]:
df.isnull().sum()

video_url        0
video_id         0
title            0
channel          0
caption          0
clean_caption    0
dtype: int64

# Cleaning Title

In [18]:
df.loc[0,['title']].values

array(['#25저지방우유먹기'], dtype=object)

In [19]:
import re

def cleaning_title(data:str) -> str:
  title = re.sub(r"#\d+", "", data).strip()
  title = title.replace("하정훈", "소아청소년과 전문의")
  
  return title.split(":")[0].strip()

In [20]:
df['title'] = df['title'].map(cleaning_title)

In [21]:
df.loc[0,['title']].values

array(['저지방우유먹기'], dtype=object)

# 학습데이터셋 컬럼 생성 

In [22]:
df['title_caption'] = df.apply(
    lambda row: "[title] " + str(row['title']) 
                + " [caption] " + str(row['clean_caption']), axis=1
)

In [23]:
df[['title', 'clean_caption', 'title_caption']].head(2)

,title,clean_caption,title_caption
0,저지방우유먹기,"안녕하세요, 소아청소년과 전문의입니다. 저지방 우유를 권하면 '우리 아이가 비만도 ...","[title] 저지방우유먹기 [caption] 안녕하세요, 소아청소년과 전문의입니다..."
2,과일먹이기,안녕하세요. 소아청소년과 전문의입니다. 과일과 채소가 건강에 좋다는 것은 누구나 아...,[title] 과일먹이기 [caption] 안녕하세요. 소아청소년과 전문의입니다. ...


# 필요없는 컬럼 삭제 

In [24]:
try:
    df.drop(columns=["channel", "video_id", "caption"], inplace=True)
except:
    pass 

In [25]:
df.head(2)

,video_url,title,clean_caption,title_caption
0,https://www.youtube.com/watch?v=-LqEr-PlCz0&li...,저지방우유먹기,"안녕하세요, 소아청소년과 전문의입니다. 저지방 우유를 권하면 '우리 아이가 비만도 ...","[title] 저지방우유먹기 [caption] 안녕하세요, 소아청소년과 전문의입니다..."
2,https://www.youtube.com/watch?v=v43KzkQWXik,과일먹이기,안녕하세요. 소아청소년과 전문의입니다. 과일과 채소가 건강에 좋다는 것은 누구나 아...,[title] 과일먹이기 [caption] 안녕하세요. 소아청소년과 전문의입니다. ...


# 학습 데이터셋 저장 

In [26]:
df.shape

(65, 4)

In [27]:
df.to_csv("./data/cleaning/nutrition.csv", index=False, header=True)